# Image Low Pass & High Pass Filters â€” FFT-Based (From Scratch)
**Same idea as 1D signal filtering, but applied to a 2D image.**

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color   # only used to load a sample image

## 2. Load a Grayscale Image

We use a built-in sample image so no file upload is needed.  
You can replace this with your own image using `plt.imread('your_image.jpg')`.

In [ ]:
# Load sample image and convert to grayscale
image = color.rgb2gray(data.astronaut())   # shape: (512, 512)

plt.figure(figsize=(5, 5))
plt.imshow(image, cmap='gray')
plt.title("Original Grayscale Image")
plt.axis('off')
plt.show()

print("Image shape:", image.shape)

## 3. Key Concept â€” 2D FFT

In 1D we did: `signal â†’ FFT â†’ mask â†’ IFFT`  
In 2D we do:  `image  â†’ FFT2 â†’ mask â†’ IFFT2`

- `np.fft.fft2(image)` â†’ converts the whole image to 2D frequency domain  
- `np.fft.fftshift(...)` â†’ moves the zero-frequency (DC) component to the **center** of the spectrum (easier to build a circular mask)  
- We build a **circular mask** centered at the middle:  
  - Low Pass  â†’ keep a circle in the center (low frequencies)  
  - High Pass â†’ keep everything **outside** the circle (high frequencies)  
- Then `np.fft.ifftshift` + `np.fft.ifft2` to get back the filtered image

## 4. Build the Circular Mask

In [ ]:
def make_circular_mask(shape, radius):
    """
    Creates a circular mask of 1s inside a circle of given radius,
    centered in an array of given shape.
    Used for Low Pass. Invert (1 - mask) for High Pass.
    """
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2   # center of the image

    # Build a grid of distances from the center
    Y, X = np.ogrid[:rows, :cols]
    dist_from_center = np.sqrt((Y - crow)**2 + (X - ccol)**2)

    # Mask = 1 inside the circle, 0 outside
    mask = (dist_from_center <= radius).astype(float)
    return mask


# Visualize the mask
radius = 50
mask = make_circular_mask(image.shape, radius)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(mask, cmap='gray')
axes[0].set_title(f"Low Pass Mask (radius={radius})\nWhite = keep, Black = remove")
axes[0].axis('off')

axes[1].imshow(1 - mask, cmap='gray')
axes[1].set_title(f"High Pass Mask (radius={radius})\nWhite = keep, Black = remove")
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 5. Low Pass Filter

In [ ]:
def low_pass_filter_2d(image, radius):
    """
    2D Low Pass Filter using FFT â€” from scratch, non-ideal.
    Keeps low frequencies (center of spectrum), removes high frequencies.
    Result: blurred / smoothed image.
    """
    # Step 1: 2D FFT
    fft_image = np.fft.fft2(image)

    # Step 2: Shift zero-frequency to center
    fft_shifted = np.fft.fftshift(fft_image)

    # Step 3: Build circular mask (keep center)
    mask = make_circular_mask(image.shape, radius)

    # Step 4: Apply mask
    fft_filtered = fft_shifted * mask

    # Step 5: Shift back and inverse FFT
    fft_back = np.fft.ifftshift(fft_filtered)
    filtered_image = np.fft.ifft2(fft_back).real

    return filtered_image


radius = 50
lp_image = low_pass_filter_2d(image, radius)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(image, cmap='gray')
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(lp_image, cmap='gray')
axes[1].set_title(f"Low Pass Filter (radius={radius})\nBlurs the image â€” removes fine details")
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 6. High Pass Filter

In [ ]:
def high_pass_filter_2d(image, radius):
    """
    2D High Pass Filter using FFT â€” from scratch, non-ideal.
    Keeps high frequencies (outer spectrum), removes low frequencies.
    Result: edges and fine details only.
    """
    # Step 1: 2D FFT
    fft_image = np.fft.fft2(image)

    # Step 2: Shift zero-frequency to center
    fft_shifted = np.fft.fftshift(fft_image)

    # Step 3: Build inverted circular mask (remove center, keep outer)
    lp_mask = make_circular_mask(image.shape, radius)
    hp_mask = 1 - lp_mask   # just flip the mask!

    # Step 4: Apply mask
    fft_filtered = fft_shifted * hp_mask

    # Step 5: Shift back and inverse FFT
    fft_back = np.fft.ifftshift(fft_filtered)
    filtered_image = np.fft.ifft2(fft_back).real

    return filtered_image


radius = 50
hp_image = high_pass_filter_2d(image, radius)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(image, cmap='gray')
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(hp_image, cmap='gray')
axes[1].set_title(f"High Pass Filter (radius={radius})\nKeeps edges and fine details only")
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 7. Full Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image, cmap='gray')
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(lp_image, cmap='gray')
axes[1].set_title("Low Pass\n(Smooth / Blur)")
axes[1].axis('off')

axes[2].imshow(hp_image, cmap='gray')
axes[2].set_title("High Pass\n(Edges / Details)")
axes[2].axis('off')

plt.suptitle(f"FFT-Based Image Filtering  |  radius = {radius}", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Effect of Different Radius Values

Try different radius values to see how the cutoff affects the result.

In [ ]:
radii = [10, 30, 80]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for i, r in enumerate(radii):
    lp = low_pass_filter_2d(image, r)
    hp = high_pass_filter_2d(image, r)

    axes[0][i].imshow(lp, cmap='gray')
    axes[0][i].set_title(f"Low Pass  |  radius={r}")
    axes[0][i].axis('off')

    axes[1][i].imshow(hp, cmap='gray')
    axes[1][i].set_title(f"High Pass  |  radius={r}")
    axes[1][i].axis('off')

plt.suptitle("Effect of Different Radius Values", fontsize=13)
plt.tight_layout()
plt.show()

## 9. Visualize the FFT Spectrum

In [ ]:
fft_shifted = np.fft.fftshift(np.fft.fft2(image))

# Log scale for better visibility (magnitudes vary greatly)
magnitude_spectrum = np.log(1 + np.abs(fft_shifted))

plt.figure(figsize=(6, 5))
plt.imshow(magnitude_spectrum, cmap='inferno')
plt.title("FFT Magnitude Spectrum (log scale)\nCenter = low freq | Edges = high freq")
plt.colorbar(label='Log Magnitude')
plt.axis('off')
plt.show()

## Summary

| | Low Pass | High Pass |
|---|---|---|
| **Keeps** | Center of spectrum (low freq) | Outer spectrum (high freq) |
| **Removes** | Outer spectrum | Center of spectrum |
| **Mask** | Circle = 1, outside = 0 | Circle = 0, outside = 1 |
| **Visual effect** | Blurs / smooths the image | Shows edges and fine details |
| **Small radius** | Very blurry | Only very sharp edges |
| **Large radius** | Slightly blurry | More details preserved |

**Same 3-step logic as 1D:**  
`FFT2 â†’ fftshift â†’ apply mask â†’ ifftshift â†’ IFFT2`

**Why non-ideal?** The circular mask has a hard edge (abrupt cutoff), causing Gibbs ringing near edges. A smooth (Gaussian) window would reduce this â€” but that would be an ideal filter.